# Deep Ensemble (5 SEED)

Cinco SEED entrenados de forma independiente por fold (semillas 42, 1, 2, 3 y 4); la predicción es la media. La dispersión entre modelos es la incertidumbre epistémica de toda la red, incluido el BiLSTM.


**Datos.** Evaluaciones 0, 1 y 2, cinco folds de test cada una (15 folds en total). Se leen los `test_predictions.npz` ya guardados: el notebook no entrena ni vuelve a inferir. Todo se calcula sobre los timesteps válidos.

**Cómo leer las tablas.** Cada tabla muestra *media ± desviación estándar* entre folds para cada evaluación (0, 1 y 2) y el promedio de las tres (los 15 folds juntos). Los gráficos tienen cuatro paneles con las mismas cuatro vistas.

**Definiciones que se repiten**

- *Cerca de eventos*: ±1 s alrededor de un huso real o predicho. Es donde ocurren los errores; el ~92% restante es fondo fácil que infla los AUROC.
- *Azar*: 0.5 para un AUROC; para un AUPRC, la proporción de positivos (tasa de error o proporción de FP/FN).
- *Aleatoria* = entropía esperada (ambigüedad de la señal); *epistémica* = MI (desacuerdo entre muestras del modelo); *total* = entropía predictiva = aleatoria + MI.
- *SEED H(p)*: la incertidumbre del SEED original sobre los mismos timesteps o eventos, como referencia.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p / 'configs' / 'default.yaml').exists()), Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from eeg_uq import report

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 250)


def show(figure):
    display(figure)
    plt.close(figure)

METHOD = 'Deep Ensemble'
REFRESH = False  # True recalcula desde los NPZ (varios minutos)
results = report.method_results(METHOD, refresh=REFRESH)
metrics = results['metrics']

## 1. Detección y calibración

F1, precisión y recall por evento (IoU ≥ 0.2, métrica principal), AUROC/AUPRC de detección por timestep y calidad de la probabilidad (NLL, Brier y ECE de la clase positiva; menor es mejor).

In [ ]:
DETECTION = ['F1 evento (%)', 'precisión (%)', 'recall (%)', 'AUROC detección', 'AUPRC detección', 'NLL', 'Brier', 'ECE clase positiva']
display(report.by_evaluation(metrics, DETECTION))
display(metrics.set_index(['evaluación', 'fold'])[DETECTION].round(4))
show(report.plot_reliability(results, METHOD))

## 2. Curvas ROC

Curva promedio de los folds. *Detección*: huso vs fondo con la probabilidad. *Errores*: timesteps mal clasificados vs bien clasificados cerca de eventos, ordenados por cada medida de incertidumbre.

In [ ]:
show(report.plot_roc(results, METHOD))

## 3. Descomposición de la incertidumbre

Entropía media en bits, en todo el registro y cerca de eventos, separada en aleatoria y epistémica (MI), con la incertidumbre de SEED como referencia.

In [ ]:
DECOMPOSITION = [c for c in metrics.columns if c.endswith('(registro)') or c.endswith('(cerca de eventos)')]
display(report.by_evaluation(metrics, DECOMPOSITION))
show(report.plot_decomposition(results, METHOD))

## 4. ¿La incertidumbre detecta los errores?

Errores por timestep cerca de eventos: AUROC y AUPRC de cada medida. El AUPRC se compara con la tasa de error (azar).

In [ ]:
ERRORS = [c for c in metrics.columns if c.startswith('AUROC errores') or c.startswith('AUPRC errores')] + ['azar errores']
display(report.by_evaluation(metrics, ERRORS))
show(report.plot_error_detection(results, METHOD))

## 5. Dónde se concentra la incertidumbre

Entropía media por zona: bordes de los husos (±0.25 s del inicio o fin real), interior del huso y fondo.

In [ ]:
zones = results['zones']
for zone in ['Borde del huso', 'Interior del huso', 'Fondo']:
    print(zone)
    display(report.by_evaluation(zones[zones['zona'] == zone], [c for c in zones.columns if c not in ('evaluación', 'fold', 'zona')]))
show(report.plot_zones(results, METHOD))

## 6. Duración del evento vs incertidumbre

ρ es la correlación de Spearman entre la duración de cada evento predicho y su incertidumbre media (MI; para SEED, su entropía). ρ < 0: los eventos cortos son más inciertos.

In [ ]:
display(report.by_evaluation(metrics, ['ρ duración-incertidumbre']))
show(report.plot_duration(results, METHOD))

## 7. Aciertos, falsos positivos y falsos negativos

- **TP / FP (predicho)**: detecciones correctas y falsas, medidas en el tramo que el modelo predijo.
- **TP / FN (real)**: husos detectados y no detectados, medidos en el tramo real del huso (un FN no tiene tramo predicho). Los pares con IoU < 0.2 cuentan como FN, igual que en el F1.

Primero las medianas de cada medida, luego qué tan bien cada medida separa FP de TP y FN de TP (AUROC y AUPRC con su azar), y al final los gráficos de cajón en percentil dentro del fold, con la línea de azar en 0.5. Comparar la *probabilidad media* entre FN y TP es casi circular (un FN es justamente un tramo con probabilidad baja).

In [ ]:
events = results['events']
print('Medianas por tipo de evento')
display(report.event_medians(events))
print('FP vs TP (tramo predicho)')
display(report.event_separation(events, 'TP (predicho)', 'FP (predicho)'))
print('FN vs TP (tramo real)')
display(report.event_separation(events, 'TP (real)', 'FN (real)'))
for figure in report.plot_event_boxes(results, METHOD):
    show(figure)

## 8. Derivar a revisión las detecciones más inciertas

Si se manda a revisión el x% de detecciones con más incertidumbre, ¿qué precisión tiene lo que se acepta automáticamente y qué parte de los falsos positivos se captura?

In [ ]:
deferral = results['deferral'].dropna(subset=['% derivado'])
for label, part in [(f'eval {e}', deferral[deferral['evaluación'] == e]) for e in report.EVALUATIONS] + [(report.MEAN_SCOPE.lower(), deferral)]:
    print(label)
    display(part.groupby(['medida', '% derivado'])[['precisión aceptado (%)', '% de FP capturados']].mean().round(1).unstack('medida'))
show(report.plot_deferral(results, METHOD))

## 9. Descartar automáticamente las detecciones más inciertas

Sin revisión humana: se eliminan las detecciones con mayor incertidumbre (MI; para SEED, entropía). Los aciertos eliminados pasan a ser falsos negativos. Para que el F1 suba, más de ~58% de lo descartado debe ser FP (con F1 ≈ 83%, el equilibrio es 1 − F1/2).

In [ ]:
rejection = results['deferral'].dropna(subset=['% descartado'])
display(rejection.groupby(['medida', '% descartado'])[['FP descartados', 'TP descartados', '% FP en lo descartado', 'cambio F1 (puntos)']].mean().round(2))
show(report.plot_rejection(results, METHOD))

## 10. Información del método

F1 por evento de cada miembro del ensamble, por fold.

In [ ]:
info = results['info']
display(info.set_index(['evaluación', 'fold']).round(3))
display(report.by_evaluation(info))

## Ejemplo temporal

Mismo segmento que el ejemplo de Laplace de la presentación: evaluación 0, fold 1, registro 261, 20 s desde el segundo 60. Paneles: señal EEG con los husos del experto sombreados, etiqueta real, predicción de SEED, predicción de Deep Ensemble y su MI en bits.

In [ ]:
import os

import h5py
import numpy as np

from eeg_uq.method_summary import load_fold

DATA_ROOT = Path(os.environ.get('EEG_DATA_ROOT', Path.home() / 'Downloads' / 'data' / 'data'))
SR = 200
EXAMPLE_EVALUATION, EXAMPLE_FOLD, EXAMPLE_SAMPLE, EXAMPLE_START_S = 0, 0, 261, 60
GREEN, BLUE = '#2ca25f', '#2a78d6'

seed_fold = load_fold(report.method_path('SEED', EXAMPLE_EVALUATION, EXAMPLE_FOLD), 'seed')
method_fold = load_fold(report.method_path(METHOD, EXAMPLE_EVALUATION, EXAMPLE_FOLD), 'sampling')
row = int(np.flatnonzero(method_fold['sample_id'] == EXAMPLE_SAMPLE)[0])
seed_row = int(np.flatnonzero(seed_fold['sample_id'] == EXAMPLE_SAMPLE)[0])
window = slice(EXAMPLE_START_S * SR, (EXAMPLE_START_S + 20) * SR)
with h5py.File(DATA_ROOT / 'moda_moda.h5', 'r') as source:
    signal = np.asarray(source['signals'][EXAMPLE_SAMPLE], dtype=float)[window]
labels = method_fold['y_true'][row][window]
times = np.arange(len(labels)) / SR
mi = method_fold['mi'][row][window]
print(f'MI en el segmento: máximo {mi.max():.4f} bits, media {mi.mean():.4f} bits')

panels = [('SEED\ny_predict', seed_fold['processed_pred'][seed_row][window], 'binario'),
          ('Deep Ens.\ny_predict', method_fold['processed_pred'][row][window], 'binario'),
          ('Deep Ens.\nMI (bits)', mi, 'curva')]
fig, axes = plt.subplots(2 + len(panels), 1, figsize=(14, 8.7), sharex=True,
                         gridspec_kw={'height_ratios': [2.6, 0.9, 0.9, 0.9, 0.9]}, constrained_layout=True)
fig.suptitle(f'{METHOD}: detección e incertidumbre — fold {EXAMPLE_FOLD + 1}, registro {EXAMPLE_SAMPLE}, segmento de 20 s')
axes[0].plot(times, signal, color='#263238', lw=0.6)
axes[0].set_ylabel('EEG (µV)')
axes[1].step(times, labels, where='post', color='black', lw=1.2)
axes[1].set(ylabel='y_true', ylim=(-0.1, 1.1))
for ax, (label, values, kind) in zip(axes[2:], panels):
    if kind == 'binario':
        ax.step(times, values, where='post', color=BLUE, lw=1.3)
        ax.set_yticks([0, 1], ['Fondo', 'Evento'])
        ax.set_ylim(-0.1, 1.1)
    else:
        ax.plot(times, values, color='#d81b60', lw=1.2)
    ax.set_ylabel(label)
edges = np.diff(np.r_[0, labels, 0])
for ax in axes:
    for begin, stop in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
        ax.axvspan(times[begin], times[min(stop, len(times) - 1)], color=GREEN, alpha=0.15, lw=0)
    ax.grid(alpha=0.25)
axes[-1].set(xlabel='Tiempo dentro del segmento (s)', xlim=(0, 20))
show(fig)

## Ejemplos temporales por tipo de evento

Segmentos de 20 s centrados en un evento del método (evaluación 0, fold 1), uno por tipo:

- **TP**: el método detectó un huso real.
- **FP**: el método detectó un huso que el experto no marcó.
- **FN**: el experto marcó un huso que el método no detectó.

Para cada tipo se muestran dos casos: uno con **MI alta** (percentil 90 de su tipo) y otro con **MI baja** (percentil 10). Las líneas naranjas punteadas marcan el evento del ejemplo; el verde, los husos del experto. Todos los paneles de MI usan la misma escala para poder compararlos. Solo se consideran eventos de al menos 0,5 s.

In [ ]:
PREFIX = 'Deep Ens.'
EXAMPLE_PERCENTILES = (('MI alta', 0.9), ('MI baja', 0.1))
ORANGE = '#eb6834'
row_of = {int(s): r for r, s in enumerate(method_fold['sample_id'])}
seed_row_of = {int(s): r for r, s in enumerate(seed_fold['sample_id'])}
length = method_fold['y_true'].shape[1]

# MI media de cada evento del método: TP y FP en el tramo predicho, FN en el tramo real.
records = []
for sample, status, p0, p1, t0, t1 in zip(method_fold['event_sample_id'], method_fold['event_status'],
                                          method_fold['event_pred_start'], method_fold['event_pred_end'],
                                          method_fold['event_true_start'], method_fold['event_true_end']):
    if status in ('TP', 'FP'):
        start, end = int(p0), int(p1)
    elif status == 'FN':
        start, end = int(t0), int(t1)
    else:
        continue
    if end - start < SR // 2:
        continue
    row = row_of[int(sample)]
    records.append({'tipo': str(status), 'registro': int(sample), 'inicio': start, 'fin': end,
                    'MI media': float(method_fold['mi'][row, start:end].mean())})
fold_events = pd.DataFrame(records)

examples = []
for kind in ('TP', 'FP', 'FN'):
    group = fold_events[fold_events['tipo'] == kind].sort_values('MI media').reset_index(drop=True)
    for label, quantile in EXAMPLE_PERCENTILES:
        pick = group.iloc[int(round(quantile * (len(group) - 1)))]
        center = (pick['inicio'] + pick['fin']) // 2
        begin = int(np.clip(center - 10 * SR, 0, length - 20 * SR))
        examples.append((kind, label, pick, slice(begin, begin + 20 * SR)))
display(pd.DataFrame([{'tipo': k, 'caso': l, 'registro': p['registro'], 'MI media (bits)': round(p['MI media'], 4),
                       'duración (s)': (p['fin'] - p['inicio']) / SR} for k, l, p, _ in examples]))

mi_max = max(method_fold['mi'][row_of[int(p['registro'])]][w].max() for _, _, p, w in examples) * 1.1
with h5py.File(DATA_ROOT / 'moda_moda.h5', 'r') as source:
    for kind, label, pick, window in examples:
        sample = int(pick['registro'])
        row, seed_row = row_of[sample], seed_row_of[sample]
        signal = np.asarray(source['signals'][sample], dtype=float)[window]
        labels = method_fold['y_true'][row][window]
        times = np.arange(len(labels)) / SR
        focus = ((pick['inicio'] - window.start) / SR, (pick['fin'] - window.start) / SR)
        rows = [('SEED\ny_predict', seed_fold['processed_pred'][seed_row][window]),
                (PREFIX + '\ny_predict', method_fold['processed_pred'][row][window])]
        fig, axes = plt.subplots(5, 1, figsize=(14, 8.2), sharex=True,
                                 gridspec_kw={'height_ratios': [2.4, 0.8, 0.8, 0.8, 1.0]}, constrained_layout=True)
        fig.suptitle(f'{METHOD} · {kind} con {label} (MI media del evento {pick["MI media"]:.3f} bits) — registro {sample}, '
                     f'{window.start / SR:.0f}–{window.stop / SR:.0f} s')
        axes[0].plot(times, signal, color='#263238', lw=0.6)
        axes[0].set_ylabel('EEG (µV)')
        axes[1].step(times, labels, where='post', color='black', lw=1.2)
        axes[1].set(ylabel='y_true', ylim=(-0.1, 1.1))
        for ax, (name, values) in zip(axes[2:4], rows):
            ax.step(times, values, where='post', color=BLUE, lw=1.3)
            ax.set_yticks([0, 1], ['Fondo', 'Evento'])
            ax.set(ylabel=name, ylim=(-0.1, 1.1))
        axes[4].plot(times, method_fold['mi'][row][window], color='#d81b60', lw=1.2)
        axes[4].set(ylabel=PREFIX + '\nMI (bits)', ylim=(0, mi_max))
        edges = np.diff(np.r_[0, labels, 0])
        for ax in axes:
            for begin, stop in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
                ax.axvspan(times[begin], times[min(stop, len(times) - 1)], color=GREEN, alpha=0.15, lw=0)
            for edge in focus:
                ax.axvline(edge, color=ORANGE, ls='--', lw=1.6)
            ax.grid(alpha=0.25)
        axes[-1].set(xlabel='Tiempo dentro del segmento (s)', xlim=(0, 20))
        show(fig)

## Hallazgos

Cifras como eval 0 / eval 1 / eval 2 / promedio de las tres (15 folds de test); un solo número es el promedio.

1. **El mejor F1 de todos los métodos:** 83.6% / 83.8% / 83.4% / 83.6% (SEED 82.7%). Supera al promedio de sus propios miembros (~82.9%) en los 15 folds, y a SEED en 14 de 15.
2. **La mejor calibración:** ECE 0.004 / 0.003 / 0.004 / 0.004 (SEED 0.006) y NLL 0.047 (SEED 0.051).
3. **La mejor incertidumbre total para detectar errores:** AUROC 0.900 / 0.899 / 0.898 / 0.899 y AUPRC 0.41, por encima de SEED (0.894).
4. **Parte epistémica de 5.8%**, concentrada en los bordes de los husos. Su MI detecta errores con AUROC 0.877 y AUPRC 0.36 (azar 0.10).
5. **Falsos positivos:** la incertidumbre total los separa con AUROC 0.86 (la mejor), y la MI con 0.78. **Falsos negativos:** la MI llega a AUROC 0.66 y AUPRC 0.27 (azar 0.17); su MI en los husos perdidos es el doble que en los aciertos. Con Neural Process, es el único que detecta parte de los errores silenciosos.
6. **Uso práctico:** derivar a revisión el 20% más incierto captura el 60% de los FP y sube la precisión de lo aceptado de 84.4% a 92.0%. Descartar por MI no mejora el F1.

**Conclusión.** El método más completo: mejora a la vez la detección, la calibración y la capacidad de señalar errores. El costo es entrenar cinco SEED por fold.